# Text Vectorization & TF-IDF Reference Guide for Logistic Regression

## 1. Methods to Convert Text into Numbers

When implementing Logistic Regression from scratch in Python, text must be converted into a numerical feature matrix \(X \in \mathbb{R}^{N \times V}\), where N is the number of documents (rows) and V is the vocabulary size (columns).

### 1.1 Binary / Multi-Hot Encoding
* **Concept:** Marks 1 if a word appears in the document and 0 otherwise, ignoring frequency.
* **Formula:**
  \[X_{i, j} = \begin{cases} 1 & \text{if word } j \in \text{document } i \\ 0 & \text{otherwise} \end{cases}\]
* **When to use:** Sanity-checking your gradient descent implementation or working with very short texts (e.g., tweets, SMS) where words rarely repeat.

### 1.2 Bag of Words (Term Frequency)
* **Concept:** Counts the raw number of times each vocabulary word appears in a document.
* **Formula:**
  \[X_{i, j} = \text{count}(w_j, d_i)\]
* **When to use:** Quick baseline models where word repetition carries signal (e.g., spam detection). *Note: Always normalize rows so large counts do not cause sigmoid overflow.*

### 1.3 N-Grams (Bigrams / Trigrams)
* **Concept:** Groups n adjacent tokens into single features before counting (e.g., `"not good"` → `"not_good"`).
* **When to use:** Sentiment analysis or intent classification where word order flips meaning.

### 1.4 TF-IDF (Term Frequency – Inverse Document Frequency) *(Recommended)*
* **Concept:** Weights words by how frequently they appear in a specific document, penalized by how common they are across all documents.
* **Mathematical Formulation (Scikit-Learn Smooth Standard):**
  \[\text{TF}(t, d) = \frac{\text{count}(t, d)}{\vert{}d\vert{}}\]
  \[\text{IDF}(t) = \ln\left(\frac{N + 1}{\text{df}(t) + 1}\right) + 1\]
  \[\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)\]
* **L2 Row Normalization:**
  \[\hat{x}_i = \frac{x_i}{\Vert{}x_i\Vert{}_2} = \frac{x_i}{\sqrt{\sum_{j=1}^{V} x_{i,j}^2}}\]
* **When to use:** The gold standard for linear models (Logistic Regression, Linear SVM) on text classification tasks. Keeps features scaled in \([0, 1]\) for smooth gradient descent convergence.

### 1.5 Feature Hashing (Hashing Trick)
* **Concept:** Uses a deterministic hash function \(h(w) \bmod V\) to map words directly to column indices without storing a vocabulary dictionary.
* **When to use:** Massive datasets (100k+ rows) or streaming text where storing a vocabulary dictionary exceeds RAM limits.

### 1.6 Pre-trained Word Embeddings (Averaged Vectors)
* **Concept:** Maps each word to a dense pre-trained vector \(e_w \in \mathbb{R}^d\) (e.g., GloVe, Word2Vec) and averages all word vectors in a document:
  \[x_i = \frac{1}{\vert{}d_i\vert{}} \sum_{w \in d_i} e_w\]
* **When to use:** Small datasets with high synonym variation where semantic similarity matters.

---

### Summary Selection Matrix

| Method | Output Type | Captures Order? | Best Use Case |
| :--- | :--- | :--- | :--- |
| **Multi-Hot Encoding** | Sparse Binary \(\{0, 1\}\) | No | Debugging gradient descent / short texts |
| **Bag of Words** | Sparse Integers \(\mathbb{Z}_{\ge 0}\) | No | Fast baseline when repetition matters |
| **N-Grams** | Sparse Matrix | Local order | Sentiment analysis (`"not_bad"`) |
| **TF-IDF + L2 Norm** | Sparse Floats \([0, 1]\) | No (unless paired with N-grams) | **Best overall for Logistic Regression** |
| **Hashing Trick** | Fixed-size Sparse Matrix | No | High-scale / low-RAM environments |
| **Averaged Embeddings** | Dense Floats \(\mathbb{R}^d\) | No | Small training sets with many synonyms |

---

## 2. Using `TfidfVectorizer` & Parameter Reference

### 2.1 Standard Implementation Pattern

```python
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    max_features=5000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    norm='l2'
)

# 1. Fit & transform ONLY on training data
X_train = vectorizer.fit_transform(train_texts).toarray()  # Shape: (N_train, V)

# 2. Transform test data using locked training vocabulary & IDF weights
X_test = vectorizer.transform(test_texts).toarray()        # Shape: (N_test, V)
```

---

### 2.2 Complete Argument Breakdown

#### A. Text Preprocessing & Tokenization
* **`lowercase`** *(bool, default=`True`)*: Converts all characters to lowercase before tokenizing.
* **`stop_words`** *(`'english'`, list, or `None`, default=`None`)*: Removes high-frequency filler words (*"the"*, *"is"*, *"and"*).
* **`token_pattern`** *(str, default=`r"(?u)\b\w\w+\b"`)*: Regex defining valid tokens. Default extracts alphanumeric words of ≥ 2 characters and strips punctuation.

#### B. Vocabulary & Context Control
* **`ngram_range`** *(tuple `(min_n, max_n)`, default=`(1, 1)`)*:
  * `(1, 1)`: Unigrams only.
  * `(1, 2)`: Unigrams + Bigrams (recommended for sentiment tasks).
* **`max_features`** *(int or `None`, default=`None`)*: Caps vocabulary size V to the top K most frequent terms across the corpus. Directly controls the dimension of your Logistic Regression weight vector \(W \in \mathbb{R}^V\).
* **`vocabulary`** *(dict or iterable, default=`None`)*: Overrides automatic vocabulary learning with a fixed mapping `{word: col_index}`.

#### C. Frequency Filtering (Noise Reduction)
* **`min_df`** *(int or float, default=`1`)*: Ignores terms appearing in fewer than `min_df` documents (int = absolute count, float = proportion of corpus). Removes typos and ultra-rare noise.
* **`max_df`** *(int or float, default=`1.0`)*: Ignores terms appearing in more than `max_df` documents. Removes corpus-specific filler words that carry no discriminative signal.

#### D. Mathematical Scaling & Normalization
* **`norm`** *(`'l2'`, `'l1'`, or `None`, default=`'l2'`)*: Normalizes each document row to unit norm. **Keep `'l2'`** so long documents and short documents produce identically scaled gradients during Logistic Regression training.
* **`use_idf`** *(bool, default=`True`)*: Enables Inverse Document Frequency re-weighting. If `False`, computes normalized Term Frequency only.
* **`smooth_idf`** *(bool, default=`True`)*: Adds 1 to numerator and denominator of IDF (\(\ln\frac{N+1}{\text{df}+1} + 1\)) to prevent division-by-zero errors.
* **`sublinear_tf`** *(bool, default=`False`)*: Replaces raw term count with \(1 + \ln(\text{tf})\) when tf > 0. Useful for long documents where a word appearing 20× should not have 20× linear weight.

---

## 3. Array Dimensions & Data Leakage Rules

1. **Guaranteed Fixed Columns (V):** Once `.fit()` or `.fit_transform()` runs on `X_train`, the column count V is permanently locked:
   * Any unseen words in `X_test` are **ignored**.
   * Missing vocabulary words in `X_test` are padded with **`0.0`**.
   * `X_train.shape[1]` is strictly guaranteed to equal `X_test.shape[1]`, ensuring matrix multiplication Z = XW + b never raises a dimension mismatch error.
2. **Preventing Data Leakage:**
   * **DO:** `vectorizer.fit_transform(X_train)` → `vectorizer.transform(X_test)`
   * **DO NOT:** Call `fit_transform()` on `X_test` or fit on the full dataset prior to train/test splitting.

---

## 4. Pure NumPy Implementation (Zero `sklearn`)

```python
import numpy as np
from collections import Counter

class ScratchTfidfVectorizer:
    def __init__(self, max_features=None):
        self.max_features = max_features
        self.vocab = {}
        self.idf = None

    def fit(self, docs):
        tokenized = [doc.lower().split() for doc in docs]
        word_counts = Counter(word for doc in tokenized for word in doc)
        
        most_common = word_counts.most_common(self.max_features)
        self.vocab = {word: idx for idx, (word, _) in enumerate(most_common)}
        
        N = len(docs)
        V = len(self.vocab)
        df = np.zeros(V)
        
        for doc in tokenized:
            for word in set(doc):
                if word in self.vocab:
                    df[self.vocab[word]] += 1
                    
        self.idf = np.log((N + 1) / (df + 1)) + 1
        return self

    def transform(self, docs):
        N = len(docs)
        V = len(self.vocab)
        tf = np.zeros((N, V))
        
        for row, doc in enumerate(docs):
            tokens = doc.lower().split()
            if not tokens:
                continue
            counts = Counter(tokens)
            for word, count in counts.items():
                if word in self.vocab:
                    tf[row, self.vocab[word]] = count / len(tokens)
                    
        tfidf = tf * self.idf
        norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return tfidf / norms

    def fit_transform(self, docs):
        return self.fit(docs).transform(docs)
```


The formulas

Let m = number of samples, X has shape (m, n), y has shape (m,).

1. Linear part: z = Xw + b

2. Sigmoid: ŷ = σ(z) = 1 / (1 + e^(-z))

3. Loss (binary cross-entropy):
J = -(1/m) · Σ [ y·log(ŷ) + (1 - y)·log(1 - ŷ) ]

4. Gradients:

dw = (1/m) · Xᵀ (ŷ - y)
db = (1/m) · Σ (ŷ - y)

5. Parameter update:

w = w - lr · dw
b = b - lr · db

6. Prediction: class = 1 if ŷ ≥ 0.5, else 0

In [6]:
import numpy as np

class LogisticRegression:
    def __init__(self,lr=0.01,n_iters=1000):
        self.lr=lr
        self.n_iters=n_iters

    def sigmoid(self,z):
        return 1/(1+np.exp(-z))

    def init_W_B(self,n_features):
        self.w=np.zeros(n_features)
        self.b=0.0

    def binary_cross_entropy_loss(self, y_actual, y_pred):
        y_pred = np.clip(y_pred, 1e-9, 1 - 1e-9)         
        return -np.mean(y_actual * np.log(y_pred)
                        + (1 - y_actual) * np.log(1 - y_pred))   



    def fit(self,X_train,y_train):
        X_train = np.array(X_train)
        y_train = np.array(y_train).astype(float).reshape(-1) 
        n_samples,n_features=X_train.shape
        self.init_W_B(n_features)

        for i in range(self.n_iters):
            #weighted sum
            z=X_train@self.w+self.b
    
            #calculate prob
            y_pred=self.sigmoid(z=z)
            
            #calculate loss(optional)
            loss = self.binary_cross_entropy_loss(y_train,y_pred)
    
            #calculate gradients
            dw=1/n_samples * np.dot(X_train.T,(y_pred-y_train))
            db=np.mean(y_pred-y_train)
    
            #update weights
            self.w=self.w-self.lr*dw
            self.b=self.b-self.lr*db

            if i % 100 == 0:
                print(f"Iteration: {i}, Loss: {loss}")


        return

    def predict_proba(self,X):
        X=np.array(X)
        z=X@self.w+self.b
            
        return self.sigmoid(z=z)


    def predict(self,X,threshold=0.5):
        return (self.predict_proba(X=X)>=threshold).astype('int')






        

In [7]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as SkLR

X, y = make_classification(n_samples=500, n_features=6, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(lr=0.1, n_iters=1000)
model.fit(X_tr, y_tr)

print("mine:   ", (model.predict(X_te) == y_te).mean())
print("sklearn:", SkLR().fit(X_tr, y_tr).score(X_te, y_te))

Iteration: 0, Loss: 0.6931471805599452
Iteration: 100, Loss: 0.2863646007803593
Iteration: 200, Loss: 0.2673643913721563
Iteration: 300, Loss: 0.2623239353295932
Iteration: 400, Loss: 0.2604790734770351
Iteration: 500, Loss: 0.2597014336526736
Iteration: 600, Loss: 0.2593474047482012
Iteration: 700, Loss: 0.2591785324790529
Iteration: 800, Loss: 0.25909552571574024
Iteration: 900, Loss: 0.25905389832621795
mine:    0.83
sklearn: 0.83
